In [1]:
import polars as pl

file_path = 'yambda/sequential-multievent-500m/sequential-multievent-500m.inter'
inter = pl.read_csv(file_path, separator='\t', has_header=True, quote_char=None, infer_schema_length=10000)
print(inter.head(10))

shape: (10, 2)
┌───────────────┬─────────────────────────────────┐
│ user_id:token ┆ item_id_list:token_seq          │
│ ---           ┆ ---                             │
│ i64           ┆ str                             │
╞═══════════════╪═════════════════════════════════╡
│ 63569         ┆ 90232 122207 129489 197686 142… │
│ 43875         ┆ 235577 262953 21989 29666 1295… │
│ 67339         ┆ 223276 237627 151110 59477 255… │
│ 71988         ┆ 180036 205505 232647 244032 13… │
│ 9765          ┆ 256469 40583 95406 42227 21627… │
│ 64904         ┆ 71904 112953 120325 152523 169… │
│ 16424         ┆ 96806 217796 43201 217796 4320… │
│ 37863         ┆ 257988 50996 261881 270951 120… │
│ 23578         ┆ 46603 192099 23224 196817 1664… │
│ 43491         ┆ 144991 235968 64579 133215 234… │
└───────────────┴─────────────────────────────────┘


In [2]:
import json

code_path = 'yambda/sequential-multievent-500m/sequential-multievent-500m.index.json'
with open(code_path, 'r') as f:
    items2codes = json.load(f)
print(items2codes['0'])

['<|a_416|>', '<|b_97|>', '<|c_77|>', '<|d_391|>']


In [3]:
inter_item_id = inter.select("item_id_list:token_seq")


In [4]:
sample = inter_item_id[0]['item_id_list:token_seq']
for i in sample:
    print(i)


90232 122207 129489 197686 142805 197686 129489 129489 90232 122207 90232 122207 90232 90232 122207 90232 90232 122207 257057 122207 90232 129489 197686 142805 171059 268442 268234 79042 91983


In [5]:
from tqdm.auto import tqdm
import json

# 1. 预处理 mapping
item_code_map = {k: "".join(v) for k, v in items2codes.items()}

# 2. 定义带有进度条的转换函数
# 使用 tqdm 监控处理进度
total_rows = inter.height
pbar = tqdm(total=total_rows, desc="Processing rows")

def transform_seq(seq_str):
    pbar.update(1)
    if not seq_str:
        return ""
    return ",".join((item_code_map[x] for x in seq_str.split()))

try:
    # 处理数据
    df_processed = inter.select(
        pl.col('item_id_list:token_seq')
        .map_elements(transform_seq, return_dtype=pl.String)
        .alias('text')
    )
    
    # 保存为 .jsonl 格式 (NDJSON)
    # 这种格式每行一个 JSON 对象，方便流式读取和检查，且不会像缩进 JSON 那样占用过多空间
    output_file = 'llama_factory_data.jsonl'
    print(f"Saving to {output_file} in JSONL format...")
    df_processed.write_ndjson(output_file)
    print("Done.")
        
finally:
    pbar.close()

/home/hongminjie/miniconda3/envs/lf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing rows:   8%|▊         | 7147/90357 [00:00<00:03, 27022.96it/s]

Processing rows: 100%|██████████| 90357/90357 [00:02<00:00, 31889.24it/s]

Saving to llama_factory_data.jsonl in JSONL format...
Done.


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast
from tokenizers import Tokenizer, models, pre_tokenizers, processors, Regex, decoders
import os
import json

class TokenExtender:
    def __init__(self, data_path, dataset, index_file=".index.json"):
        self.data_path = data_path
        self.dataset = dataset
        self.index_file = index_file
        self.indices = None
        self.new_tokens = None
        
    def _load_data(self):
        with open(os.path.join(self.data_path, self.dataset + self.index_file), 'r') as f:
            self.indices = json.load(f)
    
    def get_new_tokens(self):
        if self.new_tokens is not None:
            return self.new_tokens
            
        if self.indices is None:
            self._load_data()
        
        self.new_tokens = set()
        for index in self.indices.values():
            for token in index:
                self.new_tokens.add(token)
        self.new_tokens = sorted(list(self.new_tokens))
        
        return self.new_tokens

# 1. 定义输入和输出路径
model_path = '/home/hongminjie/models/Qwen2-1.5B'
output_dir = './yambda/model' 

print(f"Loading model from {model_path}")
model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True)

# 2. 加载新 Token
sid_index_path = "/home/hongminjie/MiniOneRec/yambda/sequential-multievent-500m/sequential-multievent-500m.index.json"
print(f"Loading index from {sid_index_path}")

token_extender = TokenExtender(
    data_path=os.path.dirname(sid_index_path),
    dataset=os.path.basename(sid_index_path).split('.')[0]
)
new_tokens = token_extender.get_new_tokens()

if new_tokens:
    print(f"Found {len(new_tokens)} tokens from dataset")
    
    # 3. 创建新的词表，只包含数据集中的token和必要的特殊token
    special_tokens_list = ['<|pad|>', '<|eos|>', '<|bos|>', '<|unk|>', ',']
    
    # 合并：特殊token在前，数据token在后
    all_tokens = special_tokens_list + new_tokens
    vocab = {token: idx for idx, token in enumerate(all_tokens)}
    
    print(f"Creating new tokenizer with {len(vocab)} tokens (including {len(special_tokens_list)} special tokens)")
    
    # 4. 使用 tokenizers 库创建一个基于词表的 tokenizer
    tokenizer_backend = Tokenizer(models.WordLevel(vocab=vocab, unk_token='<|unk|>'))
    
    # 设置预处理器：使用正则表达式匹配 <|...|> 格式的token和逗号
    tokenizer_backend.pre_tokenizer = pre_tokenizers.Split(
        pattern=Regex(r'(<\|[^|]+\|>|,)'),
        behavior='isolated',
        invert=False
    )
    
    # 设置decoder：移除tokens之间的空格
    tokenizer_backend.decoder = decoders.Replace(" ", "")
    
    # 包装成 HuggingFace tokenizer
    new_tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer_backend,
        pad_token='<|pad|>',
        eos_token='<|eos|>',
        bos_token='<|bos|>',
        unk_token='<|unk|>',
    )
    
    print(f"New tokenizer vocab size: {len(new_tokenizer)}")
    
    # 5. 调整模型 embedding 大小以匹配新词表
    print("Resizing model embeddings...")
    model.resize_token_embeddings(len(new_tokenizer))
    
    # 6. 验证
    test_text = "<|a_60|><|b_419|><|c_250|><|d_69|>,<|a_260|><|b_304|><|c_23|><|d_445|>,<|a_127|><|b_236|><|c_120|><|d_24|>"
    encoded = new_tokenizer.tokenize(test_text)
    token_ids = new_tokenizer.encode(test_text, add_special_tokens=False)
    decoded = new_tokenizer.decode(token_ids)
    
    print(f"\nVerification:")
    print(f"  Input length: {len(test_text)} chars")
    print(f"  Number of tokens: {len(encoded)}")
    print(f"  First 10 tokens: {encoded[:10]}")
    print(f"  Token IDs (first 10): {token_ids[:10]}")
    print(f"  Decoded length: {len(decoded)} chars")
    print(f"  Match original: {decoded == test_text}")
    if decoded != test_text:
        print(f"  Input:   {test_text[:50]}...")
        print(f"  Decoded: {decoded[:50]}...")
    
    # 7. 保存
    print(f"\nSaving model and tokenizer to {output_dir}...")
    model.save_pretrained(output_dir)
    new_tokenizer.save_pretrained(output_dir)
    print("Done.")
    
    # 打印词表信息
    print(f"\nTokenizer special tokens:")
    print(f"  PAD: {new_tokenizer.pad_token} (ID: {new_tokenizer.pad_token_id})")
    print(f"  EOS: {new_tokenizer.eos_token} (ID: {new_tokenizer.eos_token_id})")
    print(f"  BOS: {new_tokenizer.bos_token} (ID: {new_tokenizer.bos_token_id})")
    print(f"  UNK: {new_tokenizer.unk_token} (ID: {new_tokenizer.unk_token_id})")
else:
    print("No tokens found to add.")

/home/hongminjie/miniconda3/envs/lf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model from /home/hongminjie/models/Qwen2-1.5B
Loading index from /home/hongminjie/MiniOneRec/yambda/sequential-multievent-500m/sequential-multievent-500m.index.json
Found 2061 tokens from dataset
Creating new tokenizer with 2066 tokens (including 5 special tokens)
New tokenizer vocab size: 2066
Resizing model embeddings...

Verification:
  Input length: 106 chars
  Number of tokens: 14
  First 10 tokens: ['<|a_60|>', '<|b_419|>', '<|c_250|>', '<|d_69|>', ',', '<|a_260|>', '<|b_304|>', '<|c_23|>', '<|d_445|>', ',']
  Token IDs (first 10): [473, 870, 1195, 2018, 4, 182, 743, 1183, 1923, 4]
  Decoded length: 106 chars
  Match original: True

Saving model and tokenizer to ./yambda/model...
[2025-12-20 21:34:41,738] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/hongminjie/miniconda3/envs/lf/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/hongminjie/miniconda3/envs/lf/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


Done.

Tokenizer special tokens:
  PAD: <|pad|> (ID: 0)
  EOS: <|eos|> (ID: 1)
  BOS: <|bos|> (ID: 2)
  UNK: <|unk|> (ID: 3)


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast
test_text = "<|a_368|><|b_75|><|c_331|><|d_73|>,<|a_44|><|b_306|><|c_126|><|d_30|>,<|a_416|><|b_326|><|c_229|><|d_162|>,<|a_31|><|b_63|><|c_390|><|d_305|>,<|a_31|><|b_306|><|c_42|><|d_459|>,<|a_44|><|b_14|><|c_300|><|d_61|>,<|a_406|><|b_112|><|c_2|><|d_40|><|e_1|>,<|a_251|><|b_10|><|c_451|><|d_129|>,<|a_467|><|b_119|><|c_458|><|d_464|>,<|a_467|><|b_119|><|c_162|><|d_5|>,<|a_278|><|b_379|><|c_264|><|d_465|>,<|a_278|><|b_306|><|c_295|><|d_98|>,<|a_278|><|b_306|><|c_431|><|d_465|>,<|a_467|><|b_221|><|c_309|><|d_238|>,<|a_44|><|b_119|><|c_264|><|d_396|>,<|a_286|><|b_91|><|c_497|><|d_311|>,<|a_286|><|b_91|><|c_497|><|d_311|>,<|a_508|><|b_224|><|c_220|><|d_346|>,<|a_80|><|b_34|><|c_433|><|d_201|>,<|a_286|><|b_25|><|c_497|><|d_92|>"
tokenizer = AutoTokenizer.from_pretrained("./yambda/model")
tokenizer(test_text, return_tensors="pt")

{'input_ids': tensor([[ 301, 1001, 1285, 2023,    4,  392,  745, 1057, 1773,    4,  355,  767,
         1171, 1609,    4,  248,  988, 1350, 1768,    4,  248,  745, 1394, 1938,
            4,  392,  571, 1251, 2010,    4,  344,  530, 1250, 1884, 2057,    4,
          172,  527, 1418, 1572,    4,  411,  537, 1425, 1944,    4,  411,  537,
         1097, 2008,    4,  201,  825, 1210, 1945,    4,  201,  745, 1244, 2050,
            4,  201,  745, 1396, 1945,    4,  411,  651, 1260, 1693,    4,  392,
          537, 1210, 1868,    4,  210, 1019, 1468, 1775,    4,  210, 1019, 1468,
         1775,    4,  457,  654, 1162, 1813,    4,  495,  793, 1398, 1653,    4,
          210,  693, 1468, 2044]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,